# Burnout Prediction — EDA & Model Training

This notebook performs Exploratory Data Analysis on the Kaggle **"Are Your Employees Burning Out?"** dataset, trains two models (Linear Regression as baseline, Random Forest as primary), compares them, and saves the chosen model.

**Features used (3):** `Designation`, `Resource Allocation`, `Mental Fatigue Score`  
**Target:** `Burn Rate` (0.0–1.0)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

DATASET_PATH = Path('..') / '..' / 'archive' / 'employee_burnout_analysis-AI.xlsx'
MODEL_PATH = Path('..') / 'model' / 'burnout_model.pkl'
RANDOM_STATE = 42

print('Setup complete.')

## 1. Load the Dataset

In [ ]:
raw = pd.read_excel(DATASET_PATH, engine='openpyxl')
print(f'Shape: {raw.shape}')
raw.head(10)

In [ ]:
raw.info()

In [ ]:
raw.describe()

## 2. Missing Value Summary

In [ ]:
null_counts = raw.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
null_counts.plot(kind='barh', ax=ax)
ax.set_xlabel('Number of Missing Values')
ax.set_title('Missing Values per Column')
for i, v in enumerate(null_counts):
    ax.text(v + 20, i, f'{v} ({v/len(raw)*100:.1f}%)', va='center')
plt.tight_layout()
plt.show()

print(f'\nTotal rows: {len(raw)}')
print(null_counts.to_string())

## 3. Select and Rename Columns

In [ ]:
RENAME = {
    'Designation': 'designation',
    'Resource Allocation': 'resource_allocation',
    'Mental Fatigue Score': 'mental_fatigue_score',
    'Burn Rate': 'burn_rate',
}

FEATURES = ['designation', 'resource_allocation', 'mental_fatigue_score']
TARGET = 'burn_rate'

df = raw.rename(columns=RENAME)[FEATURES + [TARGET]].copy()
df.head()

## 4. Distribution of Burn Rate (before cleaning)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df['burn_rate'].dropna().hist(bins=40, edgecolor='black', alpha=0.7, ax=ax)
ax.set_xlabel('Burn Rate')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Burn Rate')
ax.axvline(df['burn_rate'].dropna().mean(), color='red', linestyle='--', label=f"Mean: {df['burn_rate'].dropna().mean():.3f}")
ax.axvline(df['burn_rate'].dropna().median(), color='orange', linestyle='--', label=f"Median: {df['burn_rate'].dropna().median():.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Scatter Plots — Each Feature vs Burn Rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    ax.scatter(df[feat], df[TARGET], alpha=0.15, s=10, edgecolors='none')
    ax.set_xlabel(feat)
    ax.set_ylabel('burn_rate')
    ax.set_title(f'{feat} vs burn_rate')

plt.suptitle('Feature vs Target Scatter Plots', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Data Preprocessing

In [ ]:
before = len(df)
df = df.dropna(subset=[TARGET])
print(f'Dropped {before - len(df)} rows with null target. Remaining: {len(df)}')

for col in ['resource_allocation', 'mental_fatigue_score']:
    nulls = df[col].isna().sum()
    if nulls:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'Imputed {nulls} nulls in {col} with median = {median_val}')

print(f'\nFinal null counts:')
print(df.isnull().sum())
print(f'\nFinal shape: {df.shape}')

## 8. Train / Test Split

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')

## 9. Train Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print('=== Linear Regression ===')
print(f'  MAE:  {lr_mae:.4f}')
print(f'  RMSE: {lr_rmse:.4f}')
print(f'  R²:   {lr_r2:.4f}')
print(f'\nCoefficients:')
for feat, coef in zip(FEATURES, lr.coef_):
    print(f'  {feat}: {coef:.4f}')
print(f'  intercept: {lr.intercept_:.4f}')

## 10. Train Random Forest Regressor (Primary)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print('=== Random Forest Regressor ===')
print(f'  MAE:  {rf_mae:.4f}')
print(f'  RMSE: {rf_rmse:.4f}')
print(f'  R²:   {rf_r2:.4f}')

## 11. Model Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [lr_mae, rf_mae],
    'RMSE': [lr_rmse, rf_rmse],
    'R²': [lr_r2, rf_r2],
})

comparison_styled = comparison.style.format({
    'MAE': '{:.4f}',
    'RMSE': '{:.4f}',
    'R²': '{:.4f}',
}).set_caption('Model Performance Comparison (Test Set)')

comparison_styled

## 12. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, edgecolor='black')
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Feature Importances')
for i, (val, name) in enumerate(zip(importances, importances.index)):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center')
plt.tight_layout()
plt.show()

## 13. Actual vs Predicted (Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear Regression
ax = axes[0]
ax.scatter(y_test, lr_pred, alpha=0.3, s=10, edgecolors='none')
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Burn Rate')
ax.set_ylabel('Predicted Burn Rate')
ax.set_title(f'Linear Regression (R² = {lr_r2:.4f})')
ax.legend()
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

# Random Forest
ax = axes[1]
ax.scatter(y_test, rf_pred, alpha=0.3, s=10, edgecolors='none')
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Burn Rate')
ax.set_ylabel('Predicted Burn Rate')
ax.set_title(f'Random Forest (R² = {rf_r2:.4f})')
ax.legend()
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

plt.suptitle('Actual vs Predicted — Test Set', fontsize=14)
plt.tight_layout()
plt.show()

## 14. Residual Plot (Random Forest)

In [ ]:
residuals = y_test - rf_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Residuals vs Predicted
ax = axes[0]
ax.scatter(rf_pred, residuals, alpha=0.3, s=10, edgecolors='none')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Predicted Burn Rate')
ax.set_ylabel('Residual (Actual - Predicted)')
ax.set_title('Residuals vs Predicted')

# Residual distribution
ax = axes[1]
residuals.hist(bins=40, edgecolor='black', alpha=0.7, ax=ax)
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.set_title(f'Residual Distribution (mean={residuals.mean():.4f}, std={residuals.std():.4f})')

plt.suptitle('Residual Analysis — Random Forest', fontsize=14)
plt.tight_layout()
plt.show()

## 15. Save Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf, MODEL_PATH)
print(f'Model saved to {MODEL_PATH.resolve()}')
print(f'File size: {MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB')

## Summary

| Model | MAE | RMSE | R² |
|-------|-----|------|----|
| Linear Regression | see above | see above | see above |
| **Random Forest** | see above | see above | see above |

The Random Forest model is saved and ready to be loaded by the FastAPI prediction service.